# Overlay Histograms - vessel border-border distance

Input: distance transformation map from the "07_Distances_Vessel_border-border.ijm" Fiji script

Output: Histogram of distance distribution (in um) as txt-file

In [ ]:
# 1. Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from skimage import io

## Define standard colors and legend titles

In [ ]:
# Young mice without blood loss => shades of blue
Ycolors = ['lightblue', 'deepskyblue', 'dodgerblue', 'blue', 'navy']
Ylabels = ['VK-AA498', 'VK-AA499', 'VK-AA500', 'VK-AA501', 'VK-AA502']

# Young. blood loss mice => shades of orange
Bcolors = ['peachpuff', 'sandybrown', 'darkorange', 'chocolate', 'saddlebrown']
Blabels = ['VK-AA763', 'VK-AA764', 'VK-AA765', 'VK-AA766', 'VK-AA767']

# Old mice => shades of grey
Ocolors = ['black', 'dimgrey', 'grey', 'darkgrey', 'gainsboro']
Olabels = ['VK-AA758', 'VK-AA759', 'VK-AA760', 'VK-AA761', 'VK-AA762']

## Define datasets

In [ ]:
# Young mice 
Ymouse_1 = 'VK-AA498_shaft_2376-3055_invert_skeleton_ROI_resampled_dist.tif'
Ymouse_2 = 'VK-AA499_shaft_2065-2655_invert_skeleton_ROI_resampled_dist.tif'
Ymouse_3 = 'VK-AA500_shaft_2506-3222_invert_skeleton_ROI_resampled_dist.tif'
Ymouse_4 = 'VK-AA501_shaft_2376-3055_invert_skeleton_ROI_resampled_dist.tif'
Ymouse_5 = 'VK-AA502_shaft_2341-3010_invert_skeleton_ROI_resampled_dist.tif'

In [ ]:
# Young, blood-loss mice 
Bmouse_1 = 'VK-AA763_shaft_2142-2754_invert_skeleton_ROI_resampled_dist.tif'
Bmouse_2 = 'VK-AA764_shaft_2271-2920_invert_skeleton_ROI_resampled_dist.tif'
Bmouse_3 = 'VK-AA765_shaft_2205-2835_invert_skeleton_ROI_resampled_dist.tif'
Bmouse_4 = 'VK-AA766_shaft_2096-2695_invert_skeleton_ROI_resampled_dist.tif'
Bmouse_5 = 'VK-AA767_shaft_2166-2785_invert_skeleton_ROI_resampled_dist.tif'

In [ ]:
# Aged mice 
Omouse_1 = 'VK-AA758_shaft_2740-3523_invert_skeleton_ROI_resampled_dist.tif'
Omouse_2 = 'VK-AA759_shaft_2723-3501_invert_skeleton_ROI_resampled_dist.tif'
Omouse_3 = 'VK-AA760_shaft_2866-3685_invert_skeleton_ROI_resampled_dist.tif'
Omouse_4 = 'VK-AA761_shaft_2684-3451_invert_skeleton_ROI_resampled_dist.tif'
Omouse_5 = 'VK-AA762_shaft_2905-3735_invert_skeleton_ROI_resampled_dist.tif'

## Define function to plot the overlaid density distribution

In [ ]:
# Function to plot the density histograms all in one plot using numpy bincount
def plot_using_bincount(datasets, colors=None, labels=None, title="Histogram", xlabel="distance [µm]", ylabel="Frequency"):
    plt.figure(figsize=(10, 6))
    
    bin_indices = np.arange(1, 255) # attention: distance map is given in pixel and as half-distance
    bin_indices_um = bin_indices * 2.6 * 2 # convert to um and to full distance
    
    for i, data in enumerate(datasets):
        # Exclude zeros from the current dataset
        data = data[data != 0]
        
        # Get number of non-zero voxel
        num_nonzero = np.count_nonzero(data)
        
        # Count occurrences in each bin using np.bincount and normalize to number of occurences
        bin_counts = np.bincount(data, minlength=254)/num_nonzero
        
        # Plot using np.arange for x-axis (bin edges) and np.bincount for y-axis (frequencies)
        plt.plot(bin_indices_um[0:50], bin_counts[0:50], color=colors[i] if colors else None, label=labels[i] if labels else None, lw=2)

    # Add labels and title
    plt.xlabel(xlabel, fontsize=14, fontweight='bold')
    plt.ylabel(ylabel, fontsize=14, fontweight='bold')
    plt.title(title, fontsize=16, fontweight='bold')

    # Add legend if labels are provided
    if labels:
        plt.legend()

    # Show the plot
    plt.show()

# Function to save the density histograms
def save_density_histograms_to_csv(datasets, csv_filename="combined_density_histograms.csv"):
    
    # Create an empty list to store histogram dataframes
    dfs = []
    
    for i, data in enumerate(datasets):
        # Exclude zeros from the current dataset
        data = data[data != 0]
        
        # Compute density histogram values
        bin_indices = np.arange(1, 255)
        bin_indices_um = bin_indices * 2.6 * 2 # convert to um and to full distance
        values = np.bincount(data, minlength=254)
        
        # Create a DataFrame for the current dataset's histogram
        df = pd.DataFrame({'Bin Edges': bin_indices_um, f'Density Values_{i}': values})  # Add a column with density values for the current datasets
        
        # Append the DataFrame to the list
        dfs.append(df)
    
    # Merge all the individual dataframes on the 'Bin Edges' column
    merged_df = dfs[0]  # Start with the first DataFrame
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on='Bin Edges', how='outer')
    
    # Save the merged DataFrame to the specified CSV file
    merged_df.to_csv(csv_filename, index=False)

## Young mice dataset

In [ ]:
# Load young mice data sets
Ydatasets = [Ymouse_1, Ymouse_2, Ymouse_3, Ymouse_4, Ymouse_5]
Yall_halfdistance= list()

for dataset in Ydatasets:
    img = io.imread(dataset)
    Yall_halfdistance.append(img)

# Note: only the peak region is plotted (from 0 -50 µm)
#plot_using_bincount(Yall_halfdistance, colors=Ycolors, labels=Ylabels, title='Young Mice (LC) - knee')

## Young, blood loss mice dataset

In [ ]:
# Load young, blood loss mice data sets
Bdatasets = [Bmouse_1, Bmouse_2, Bmouse_3, Bmouse_4, Bmouse_5]
Ball_halfdistance = list()

for dataset in Bdatasets:
    img = io.imread(dataset)
    Ball_halfdistance.append(img)

# Note: only the peak region is plotted (from 0 -50 µm)
#plot_using_bincount(Ball_halfdistance, colors=Bcolors, labels=Blabels, title='Young, blood loss Mice (LC) - knee')

## Aged mice

In [ ]:
# Load aged loss mice data sets
Odatasets = [Omouse_1, Omouse_2, Omouse_3, Omouse_4, Omouse_5]
Oall_halfdistance = list()

for dataset in Odatasets:
    img = io.imread(dataset)
    Oall_halfdistance.append(img)

# Note: only the peak region is plotted (from 0 -50 µm)
#plot_using_bincount(Oall_halfdistance, colors=Ocolors, labels=Olabels, title='Aged Mice (LC) - knee')

## Save histograms (density)

In [ ]:
save_density_histograms_to_csv(Ball_halfdistance, csv_filename="Blood-let(mid-knee2).csv")
save_density_histograms_to_csv(Yall_halfdistance, csv_filename="Young(mid-knee2).csv")
save_density_histograms_to_csv(Oall_halfdistance, csv_filename="Old(mid-knee2).csv")